# WZML-X en Google Colab

Instala las dependencias nativas y Python, solicita las credenciales sin mostrarlas, inicia el bot y publica la UI web con un túnel temporal. Ejecuta las celdas **en orden**.

> **Importante:** Colab es efímero y puede desconectarse en cualquier momento. Úsalo para pruebas/sesiones temporales, no como alojamiento 24/7. Activa el acceso a Internet del runtime. `DATABASE_URL` (MongoDB Atlas) es opcional para arrancar, pero necesaria para conservar ajustes/estado entre sesiones.

No compartas el notebook con credenciales. Los secretos se piden con entrada oculta o se leen de Colab Secrets; `config.py` se crea solo en el disco temporal del runtime. La carpeta de descargas se borra al iniciar, igual que en el despliegue normal.

La integración MEGA usa un binding nativo que no viene en `requirements.txt`. Si no está disponible o es incompatible, el bot se inicia con MEGA desactivado; las demás funciones continúan. No instales `mega`/`mega.py` de PyPI como sustituto.


In [ ]:
# Repositorio/branch a ejecutar. Cambia estos dos valores si usas un fork.
REPO_URL = "https://github.com/leomarmol03/WZML-X.git"
BRANCH = "wzv3"

from pathlib import Path
import os
import subprocess

ROOT = Path("/content/WZML-X")
if not (ROOT / "bot").is_dir():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(f"{ROOT} existe pero no parece una copia de WZML-X; elige otra ruta o vacíala manualmente.")
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
print(f"Repositorio listo: {ROOT} (branch {BRANCH})")


## 1. Dependencias del sistema

Colab no usa Docker ni systemd. Por eso esta celda instala los ejecutables que el bot normalmente recibe de su imagen Docker y comprueba que `7z` esté disponible. JDownloader y SABnzbd quedan desactivados en la configuración para reducir consumo; el Java Runtime se instala para que puedas habilitar JDownloader manualmente si lo necesitas.

La integración de MEGA no se instala aquí: depende de un binding nativo específico incluido en la imagen Docker, cuya compilación no se ha validado en Colab. El bot la deshabilitará automáticamente si falta.


In [ ]:
import shutil
import subprocess

subprocess.run(["apt-get", "update", "-qq"], check=True)
system_packages = [
    "aria2", "qbittorrent-nox", "ffmpeg", "rclone",
    "curl", "ca-certificates", "git", "default-jre-headless",
    "cpulimit", "libmagic1", "libmagic-dev", "libattr1-dev",
    "build-essential",
]
subprocess.run(["apt-get", "install", "-y", "-qq", "--no-install-recommends", *system_packages], check=True)

# Ubuntu/Colab puede ofrecer p7zip-full o 7zip según la imagen base.
if not shutil.which("7z"):
    result = subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if result.returncode != 0 and not shutil.which("7z"):
        subprocess.run(["apt-get", "install", "-y", "-qq", "7zip"], check=True)
if not shutil.which("7z"):
    alternative = shutil.which("7zz") or shutil.which("7za")
    if alternative:
        Path("/usr/local/bin/7z").symlink_to(alternative)

# Herramientas opcionales para extraer archivos RAR y consultar metadatos.
subprocess.run(["apt-get", "install", "-y", "-qq", "unrar-free", "mediainfo"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

required_binaries = ("aria2c", "qbittorrent-nox", "ffmpeg", "ffprobe", "rclone", "7z")
missing = [name for name in required_binaries if not shutil.which(name)]
if missing:
    raise RuntimeError("Faltan ejecutables del sistema: " + ", ".join(missing))
print("Dependencias del sistema verificadas:", ", ".join(required_binaries))


## 2. Dependencias de Python y comprobación

Se instala el `requirements.txt` del proyecto en el intérprete de este runtime (el mismo que usará el proceso del bot). La compilación también detecta errores de sintaxis antes de intentar iniciar Telegram.

El binding nativo de MEGA no se incluye en los requisitos Python. El chequeo del módulo confirma que los imports opcionales del bot no bloquean el arranque cuando ese SDK no está presente.


In [ ]:
import compileall
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", "requirements.txt", "uv"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
if not compileall.compile_dir(str(ROOT / "bot"), quiet=1) or not compileall.compile_dir(str(ROOT / "web"), quiet=1):
    raise RuntimeError("Hay un error de sintaxis en el código del bot/web")
subprocess.run([sys.executable, "-c", "from bot.core.config_manager import BinConfig, Config; from bot.helper.ext_utils.mega_sdk import MEGA_SDK_AVAILABLE; print(BinConfig.ARIA2_NAME, BinConfig.QBIT_NAME, BinConfig.FFMPEG_NAME, BinConfig.RCLONE_NAME, f'MEGA SDK available: {MEGA_SDK_AVAILABLE}; disabled by default: {Config.DISABLE_MEGA}')"], cwd=ROOT, check=True)
print("Dependencias Python y comprobación de sintaxis: OK")


## 3. Configuración privada

Puedes crear secretos en **Colab → Secrets** con los nombres indicados. Si no existen, el notebook los solicita de forma oculta. `OWNER_ID` es tu ID numérico de Telegram; obtén `TELEGRAM_API` y `TELEGRAM_HASH` en `my.telegram.org`. La URI de MongoDB se puede dejar vacía, aunque sin ella la configuración no sobrevive al runtime.


In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def read_secret(name, prompt, optional=False):
    value = ""
    if userdata is not None:
        try:
            value = userdata.get(name) or ""
        except Exception:
            pass
    if not value:
        value = getpass(prompt)
    value = str(value).strip()
    if not value and not optional:
        raise ValueError(f"{name} es obligatorio")
    return value

BOT_TOKEN = read_secret("BOT_TOKEN", "BOT_TOKEN de @BotFather: ")
OWNER_ID = int(read_secret("OWNER_ID", "Tu OWNER_ID (número de Telegram): "))
TELEGRAM_API = int(read_secret("TELEGRAM_API", "TELEGRAM_API (ID numérico de my.telegram.org): "))
TELEGRAM_HASH = read_secret("TELEGRAM_HASH", "TELEGRAM_HASH de my.telegram.org: ")
DATABASE_URL = read_secret("DATABASE_URL", "MongoDB Atlas URI (opcional, Enter para omitir): ", optional=True)
if OWNER_ID <= 0 or TELEGRAM_API <= 0 or ":" not in BOT_TOKEN:
    raise ValueError("OWNER_ID, TELEGRAM_API o BOT_TOKEN no tienen un formato válido")

# Desactiva servicios pesados no necesarios en Colab. Aria2/qBittorrent siguen activos.
config_values = {
    "BOT_TOKEN": BOT_TOKEN,
    "OWNER_ID": OWNER_ID,
    "TELEGRAM_API": TELEGRAM_API,
    "TELEGRAM_HASH": TELEGRAM_HASH,
    "DATABASE_URL": DATABASE_URL,
    "BASE_URL": "",
    "DISABLE_MEGA": False,  # Se fuerza a True si falta el binding nativo compatible.
    "DISABLE_TORRENTS": False,
    "DISABLE_STREAM": False,
    "DISABLE_JD": True,
    "DISABLE_NZB": True,
    "BOT_MAX_TASKS": 2,
    "USER_MAX_TASKS": 2,
}
config_text = "\n".join(f"{key} = {value!r}" for key, value in config_values.items()) + "\n"
config_path = ROOT / "config.py"
config_path.write_text(config_text, encoding="utf-8")
config_path.chmod(0o600)
print("config.py creado dentro del runtime. No se muestran ni se imprimen las credenciales.")


## 4. Iniciar el bot

Este arranque ejecuta `python -m bot` directamente: no ejecuta `update.py`, no necesita Docker y no borra los archivos del repositorio. El proceso queda en segundo plano para poder usar otra celda para la UI y los logs.


In [ ]:
import os
import signal
import subprocess
import sys
import time

previous_bot = globals().get("bot_process")
if previous_bot is not None and previous_bot.poll() is None:
    raise RuntimeError("El bot ya está ejecutándose. Detén el proceso antes de volver a iniciarlo.")
BOT_LOG = ROOT / "colab_bot.log"
TUNNEL_URL_FILE = ROOT / ".colab_tunnel_url"
TUNNEL_URL_FILE.write_text("", encoding="utf-8")
bot_env = os.environ.copy()
bot_env.update({
    "PORT": "8080",
    "TUNNEL_URL_FILE": str(TUNNEL_URL_FILE),
    "WZML_DOWNLOAD_DIR": str(ROOT / "downloads"),
    "WZML_JDOWNLOADER_DIR": str(ROOT / "JDownloader"),
    "PYTHONUNBUFFERED": "1",
})
with BOT_LOG.open("w", encoding="utf-8") as log_file:
    bot_process = subprocess.Popen([sys.executable, "-m", "bot"], cwd=ROOT, env=bot_env, stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)

def show_bot_log(lines=30):
    if BOT_LOG.exists():
        print("\n".join(BOT_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:]))

deadline = time.monotonic() + 300
while time.monotonic() < deadline:
    if bot_process.poll() is not None:
        show_bot_log(80)
        raise RuntimeError(f"El bot terminó con código {bot_process.returncode}; revisa el log anterior.")
    log_text = BOT_LOG.read_text(encoding="utf-8", errors="replace") if BOT_LOG.exists() else ""
    if "WZ Client(s) & Services Started" in log_text:
        break
    time.sleep(2)
else:
    print("El proceso sigue iniciando; este runtime puede tardar más. Últimas líneas:")
show_bot_log(35)
print("Bot iniciado. Mantén el runtime de Colab activo mientras lo uses.")


## 5. UI web y selector de archivos (opcional, recomendado)

Colab no expone puertos entrantes. Esta celda intenta crear un Cloudflare Quick Tunnel gratuito y temporal para el puerto 8080. El enlace se comparte con el bot automáticamente para que los botones de selección de archivos funcionen. Si no se puede descargar/ejecutar `cloudflared`, se intenta usar el proxy de Colab como alternativa.


In [ ]:
import os
import re
import shutil
import signal
import subprocess
import time

previous_tunnel = globals().get("tunnel_process")
if previous_tunnel is not None and previous_tunnel.poll() is None:
    os.killpg(previous_tunnel.pid, signal.SIGTERM)
    try:
        previous_tunnel.wait(timeout=10)
    except subprocess.TimeoutExpired:
        os.killpg(previous_tunnel.pid, signal.SIGKILL)
TUNNEL_LOG = ROOT / "colab_tunnel.log"
tunnel_url = ""
tunnel_process = None
if not shutil.which("cloudflared"):
    cloudflared_deb = Path("/tmp/cloudflared-linux-amd64.deb")
    download = subprocess.run(["curl", "-fL", "--retry", "2", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb", "-o", str(cloudflared_deb)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if download.returncode == 0:
        installed = subprocess.run(["dpkg", "-i", str(cloudflared_deb)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if installed.returncode != 0:
            subprocess.run(["apt-get", "-f", "install", "-y", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.run(["dpkg", "-i", str(cloudflared_deb)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if shutil.which("cloudflared"):
    with TUNNEL_LOG.open("w", encoding="utf-8") as tunnel_log:
        tunnel_process = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        text = TUNNEL_LOG.read_text(encoding="utf-8", errors="replace") if TUNNEL_LOG.exists() else ""
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
        if match:
            tunnel_url = match.group(0).rstrip("/")
            break
        if tunnel_process.poll() is not None:
            break
        time.sleep(2)
if not tunnel_url:
    try:
        from google.colab.output import eval_js
        tunnel_url = eval_js("google.colab.kernel.proxyPort(8080)") or ""
    except Exception as exc:
        print(f"No se pudo crear el túnel de Cloudflare ni el proxy de Colab: {exc}")
if tunnel_url:
    TUNNEL_URL_FILE.write_text(tunnel_url, encoding="utf-8")
    print("UI web del bot:", tunnel_url)
    print("El bot aplicará esta URL en unos segundos; el túnel solo vive mientras siga activo este runtime.")
else:
    print("La UI externa no quedó expuesta. El bot de Telegram puede seguir funcionando, pero los selectores web no estarán disponibles.")
    if TUNNEL_LOG.exists():
        print("\n".join(TUNNEL_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]))


## Logs y apagado

Vuelve a ejecutar esta celda para consultar las últimas líneas. La celda de apagado termina los procesos del bot y del túnel; no borra `config.py`. Al terminar el runtime, Colab elimina el disco temporal.


In [ ]:
show_bot_log(60)


In [ ]:
def stop_colab_process(process, label):
    if process is None or process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait(timeout=10)
    except ProcessLookupError:
        pass
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
    print(f"{label}: detenido")

stop_colab_process(globals().get("tunnel_process"), "Túnel")
stop_colab_process(globals().get("bot_process"), "Bot")
